# Pronostico de Demanda de Bicicletas con Machine Learning

## Forecasting Recursivo con Skforecast, LightGBM y Variables Exogenas

> **Dataset:** [Bike Sharing Dataset — UCI Machine Learning Repository](https://archive.uci.edu/dataset/275/bike+sharing+dataset)

---

Este cuadernillo es una guia completa de **forecasting de series de tiempo con Machine Learning**.
Trabajamos con registros horarios de alquiler de bicicletas en Washington D.C. (2011-2012)
y construimos progresivamente modelos cada vez mas sofisticados.

### Flujo de trabajo

```
Datos → Imputacion → EDA → Baseline → Univariado → Feature Eng → Exogenas → Optimizacion → SHAP
```

### Contenido

| # | Seccion | Concepto clave |
|---|---------|----------------|
| 1 | Carga y exploracion | `pandas`, `plotly` |
| 2 | Indice temporal y imputacion | `asfreq`, interpolacion |
| 3 | Analisis exploratorio (EDA) | Patrones horarios y estacionales |
| 4 | Division Train/Val/Test | Respeto del orden temporal |
| 5 | Modelo Baseline | `ForecasterEquivalentDate` |
| 6 | Forecaster Univariado | `ForecasterRecursive`, lags, window features |
| 7 | Feature Engineering | Variables ciclicas, calendarias |
| 8 | Forecaster con Exogenas | Variables meteorologicas |
| 9 | Optimizacion Bayesiana | `bayesian_search_forecaster`, Optuna |
| 10 | Evaluacion final | MAE, RMSE, MAPE, R2 |
| 11 | Analisis SHAP | Interpretabilidad del modelo |
| 12 | Predicciones futuras | Simulacion en produccion |

---
## 0. Instalacion de Librerias

In [ ]:
# Instalar librerias (ejecutar solo si no estan instaladas)
# ==============================================================================
%pip install -q skforecast plotly xgboost catboost lightgbm scikit-learn shap feature-engine optuna

In [ ]:
# ==============================================================================
# IMPORTACION DE LIBRERIAS
# ==============================================================================

# Procesamiento de datos
import numpy as np
import pandas as pd
from feature_engine.creation import CyclicalFeatures

# Visualizacion
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = 'seaborn'
plt.style.use('seaborn-v0_8-darkgrid')

# Machine Learning
import xgboost
import lightgbm
import catboost
import sklearn
import shap
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    root_mean_squared_error,
    r2_score
)

# Forecasting con Skforecast
import skforecast
from skforecast.recursive import ForecasterEquivalentDate, ForecasterRecursive
from skforecast.model_selection import TimeSeriesFold
from skforecast.model_selection import bayesian_search_forecaster, backtesting_forecaster
from skforecast.preprocessing import RollingFeatures

import warnings
warnings.filterwarnings('ignore')

# Versiones instaladas
print('=' * 50)
print('  Versiones de librerias instaladas')
print('=' * 50)
print(f'  skforecast  : {skforecast.__version__}')
print(f'  scikit-learn: {sklearn.__version__}')
print(f'  lightgbm    : {lightgbm.__version__}')
print(f'  xgboost     : {xgboost.__version__}')
print(f'  catboost    : {catboost.__version__}')
print(f'  pandas      : {pd.__version__}')
print(f'  numpy       : {np.__version__}')

---
## 1. Carga y Exploracion del Dataset

### Descripcion de las variables

| Variable | Tipo | Descripcion |
|----------|------|-------------|
| `cnt` | Entero | Total de usuarios (casual + registrados) — **Variable objetivo** |
| `temp` | Continua [0,1] | Temperatura normalizada |
| `atemp` | Continua [0,1] | Sensacion termica normalizada |
| `hum` | Continua [0,1] | Humedad relativa normalizada |
| `windspeed` | Continua [0,1] | Velocidad del viento normalizada |
| `weathersit` | Categorica (1-4) | Situacion meteorologica |
| `holiday` | Binaria | 1 si es dia festivo |
| `season` | Categorica (1-4) | Estacion del año |
| `hr` | Entero (0-23) | Hora del dia |

### Codigos de `weathersit`
- **1**: Despejado / Pocas nubes
- **2**: Neblina / Nublado
- **3**: Lluvia ligera / Nevada ligera
- **4**: Lluvia intensa / Tormenta

In [ ]:
# Cargar dataset desde GitHub
# ==============================================================================
url = 'https://github.com/Wilsonsr/Series-de-Tiempo/raw/main/Data/hour.csv'
df_raw = pd.read_csv(url)

print(f'Filas    : {df_raw.shape[0]:,}')
print(f'Columnas : {df_raw.shape[1]}')
print(f'Columnas : {df_raw.columns.tolist()}')
print()
df_raw.head()

In [ ]:
# Estadisticas descriptivas
# ==============================================================================
print('Tipos de datos:')
print(df_raw.dtypes)
print()
df_raw[['cnt', 'temp', 'atemp', 'hum', 'windspeed']].describe().round(3)

---
## 2. Preparacion del Indice Temporal

Para que Skforecast pueda trabajar correctamente con la serie, necesitamos un
**DatetimeIndex** con frecuencia fija (`'h'` = horaria).

**Pasos:**
1. Combinar columna `dteday` (fecha) con `hr` (hora) para formar un datetime completo
2. Establecer el indice con `set_index()`
3. Aplicar `asfreq('h')` para forzar frecuencia horaria continua

> **Nota importante:** `asfreq('h')` introduce filas nuevas para las horas que no
> estaban en el CSV original (horas 'faltantes'). Estas filas tendran `NaN` en
> todas sus columnas y seran imputadas en la siguiente seccion.

In [ ]:
# Construir indice temporal horario
# ==============================================================================
df = df_raw.copy()

# Combinar fecha + hora
df['dteday'] = pd.to_datetime(df['dteday'], format='%Y-%m-%d')
df['dteday'] = df['dteday'] + pd.to_timedelta(df['hr'], unit='h')

# Ordenar y establecer como indice
df = df.sort_values('dteday').set_index('dteday')
df.index.name = 'datetime'

# Aplicar frecuencia horaria continua
df = df.asfreq('h')

# Diagnostico
n_expected = len(pd.date_range(df.index.min(), df.index.max(), freq='h'))
n_missing  = df.isna().any(axis=1).sum()

print(f'Rango temporal      : {df.index.min()} --> {df.index.max()}')
print(f'Frecuencia          : {df.index.freq}')
print(f'Horas esperadas     : {n_expected:,}')
print(f'Filas en DataFrame  : {len(df):,}')
print(f'Registros en CSV    : {len(df_raw):,}')
print(f'Horas a imputar     : {n_missing} filas con NaN')

---
## 3. Seleccion de Variables e Imputacion

### Por que imputar?

Al aplicar `asfreq('h')`, se crean filas para horas que no existian en el dataset
original. Estas horas quedan con `NaN` en todas sus columnas.
**Skforecast no acepta valores faltantes** en la serie objetivo ni en las variables
exogenas, por lo que debemos imputarlos antes de modelar.

### Estrategia de imputacion

| Tipo | Variables | Metodo | Justificacion |
|------|-----------|--------|---------------|
| Continuas | `users`, `temp`, `atemp`, `hum`, `windspeed` | Interpolacion temporal | Cambio gradual entre horas |
| Discretas | `holiday`, `weathersit` | Forward fill | La condicion no cambia entre horas |
| Conteo | `users` | Interpolacion + redondeo | Debe ser entero |

> **Truco:** Siempre usar `.copy()` al extraer un subconjunto de columnas de un
> DataFrame para evitar el `SettingWithCopyWarning` de pandas.

In [ ]:
# Seleccionar variables relevantes con .copy() para evitar SettingWithCopyWarning
# ==============================================================================
data = df[['cnt', 'holiday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']].copy()
data.rename(columns={'cnt': 'users'}, inplace=True)

# Diagnostico ANTES de imputar
print('=== Valores faltantes antes de imputacion ===')
missing_before = data.isna().sum()
print(missing_before)
print(f'Total filas con algun NaN: {data.isna().any(axis=1).sum()}')

In [ ]:
# Imputacion de valores faltantes
# ==============================================================================

# Variables continuas: interpolacion temporal (lineal por tiempo)
cols_continuas = ['users', 'temp', 'atemp', 'hum', 'windspeed']
for col in cols_continuas:
    data[col] = data[col].interpolate(method='time')
    data[col] = data[col].ffill().bfill()  # Seguridad para extremos

# Variable objetivo: es un conteo -> redondear a entero
data['users'] = data['users'].clip(lower=0).round().astype(int)

# Variables discretas: arrastre hacia adelante (la condicion persiste)
cols_discretas = ['holiday', 'weathersit']
for col in cols_discretas:
    data[col] = data[col].ffill().bfill()
    data[col] = data[col].round().astype(int)

# Verificacion DESPUES de imputar
print('=== Valores faltantes despues de imputacion ===')
print(data.isna().sum())
print()
print(f'Shape final: {data.shape}')
print(f'Rango de usuarios: [{data["users"].min()}, {data["users"].max()}]')
data.head(10)

---
## 4. Analisis Exploratorio de Datos (EDA)

Antes de modelar siempre debemos explorar la serie para entender:

- **Tendencia**: Crecimiento sostenido de la demanda entre 2011 y 2012
- **Estacionalidad diaria**: Picos a las 8am y 5-6pm en dias laborables
- **Estacionalidad semanal**: Diferente patron entre semana y fin de semana
- **Estacionalidad anual**: Mayor demanda en primavera/verano

Estos patrones nos guiaran al elegir los **lags** y **variables exogenas**.

In [ ]:
# Visualizacion de la serie temporal completa
# ==============================================================================
fig, axes = plt.subplots(3, 1, figsize=(14, 11))

# --- Serie completa ---
data['users'].plot(ax=axes[0], color='steelblue', linewidth=0.4, alpha=0.85)
data['users'].resample('D').mean().plot(ax=axes[0], color='orange',
    linewidth=1.5, alpha=0.9, label='Media diaria')
axes[0].set_title('Serie temporal completa: Demanda horaria de bicicletas', fontsize=13)
axes[0].set_ylabel('Usuarios')
axes[0].set_xlabel('')
axes[0].legend(['Datos horarios', 'Media diaria'])

# --- Promedio por hora del dia ---
hourly_avg = data.groupby(data.index.hour)['users'].mean()
axes[1].bar(hourly_avg.index, hourly_avg.values, color='steelblue',
            alpha=0.75, edgecolor='white')
axes[1].set_title('Demanda promedio por hora del dia', fontsize=13)
axes[1].set_xlabel('Hora del dia')
axes[1].set_ylabel('Usuarios promedio')
axes[1].set_xticks(range(24))
axes[1].axvline(8,  color='red', linestyle='--', linewidth=1, alpha=0.7, label='Pico manana (8h)')
axes[1].axvline(17, color='green', linestyle='--', linewidth=1, alpha=0.7, label='Pico tarde (17h)')
axes[1].legend(fontsize=9)

# --- Promedio por dia de la semana ---
dias = ['Lunes', 'Martes', 'Miercoles', 'Jueves', 'Viernes', 'Sabado', 'Domingo']
weekly_avg = data.groupby(data.index.dayofweek)['users'].mean()
colores_dias = ['steelblue'] * 5 + ['coral', 'coral']
axes[2].bar(range(7), weekly_avg.values, color=colores_dias, alpha=0.8, edgecolor='white')
axes[2].set_title('Demanda promedio por dia de la semana', fontsize=13)
axes[2].set_xticks(range(7))
axes[2].set_xticklabels(dias, rotation=20)
axes[2].set_ylabel('Usuarios promedio')
p1 = mpatches.Patch(color='steelblue', label='Dia laborable')
p2 = mpatches.Patch(color='coral', label='Fin de semana')
axes[2].legend(handles=[p1, p2], fontsize=9)

plt.suptitle('Analisis Exploratorio: Patrones de la Serie', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Mapa de calor: demanda promedio por hora y dia de semana
# ==============================================================================
pivot = data.pivot_table(
    values  = 'users',
    index   = data.index.hour,
    columns = data.index.dayofweek,
    aggfunc = 'mean'
).round(1)
pivot.columns = ['Lun', 'Mar', 'Mie', 'Jue', 'Vie', 'Sab', 'Dom']
pivot.index.name = 'Hora'

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Mapa de calor
im = axes[0].imshow(pivot, aspect='auto', cmap='YlOrRd')
axes[0].set_xticks(range(7))
axes[0].set_xticklabels(pivot.columns)
axes[0].set_yticks(range(0, 24, 2))
axes[0].set_yticklabels(range(0, 24, 2))
axes[0].set_title('Mapa de calor: Demanda media\n(Hora vs Dia de semana)', fontsize=12)
axes[0].set_xlabel('Dia de la semana')
axes[0].set_ylabel('Hora del dia')
plt.colorbar(im, ax=axes[0], label='Usuarios promedio')

# Demanda por mes
monthly_avg = data.groupby(data.index.month)['users'].mean()
meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
         'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
axes[1].bar(range(1, 13), monthly_avg.values, color='mediumseagreen',
            alpha=0.8, edgecolor='white')
axes[1].set_title('Demanda promedio por mes\n(Estacionalidad anual)', fontsize=12)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(meses, rotation=30)
axes[1].set_ylabel('Usuarios promedio')

plt.tight_layout()
plt.show()

print('Observaciones clave:')
print('  - Picos de demanda a las 8h y 17-18h en dias laborables (commuters)')
print('  - Fines de semana: curva mas plana y centrada al mediodia')
print('  - Mayor demanda en verano (Jun-Sep) y menor en invierno (Ene-Feb)')

In [ ]:
# Correlacion con variables meteorologicas
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribucion de la variable objetivo
data['users'].hist(bins=60, ax=axes[0], color='steelblue', alpha=0.75, edgecolor='white')
axes[0].axvline(data['users'].mean(), color='red', linestyle='--',
                linewidth=2, label=f'Media: {data["users"].mean():.0f}')
axes[0].axvline(data['users'].median(), color='orange', linestyle='--',
                linewidth=2, label=f'Mediana: {data["users"].median():.0f}')
axes[0].set_title('Distribucion de la demanda (users)', fontsize=12)
axes[0].set_xlabel('Numero de usuarios')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Matriz de correlacion
corr_cols = ['users', 'temp', 'atemp', 'hum', 'windspeed']
corr = data[corr_cols].corr()
im2 = axes[1].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr_cols)))
axes[1].set_yticks(range(len(corr_cols)))
axes[1].set_xticklabels(corr_cols, rotation=45, ha='right')
axes[1].set_yticklabels(corr_cols)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        axes[1].text(j, i, f'{corr.iloc[i, j]:.2f}',
                     ha='center', va='center', fontsize=10,
                     color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')
axes[1].set_title('Correlacion de Pearson', fontsize=12)
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print('Correlaciones con users:')
print(corr['users'].sort_values(ascending=False).drop('users').round(3))

---
## 5. Division Train / Validacion / Test

En series de tiempo **nunca** se puede mezclar temporalmente los datos ni usar
validacion cruzada aleatoria. La division debe respetar el orden cronologico:

```
2011-01-01 ──────────── 2012-03-30 ────── 2012-07-31 ───── 2012-12-31
│                                │                  │               │
│        TRAIN (62%)             │   VALIDACION     │    TEST (21%) │
│     (para entrenar)            │     (17%)        │  (evaluacion) │
│                                │  (hiperparams)   │    FINAL      │
└────────────────────────────────┴──────────────────┴───────────────┘
```

**Regla de oro:** El conjunto de **Test NUNCA** se toca hasta la evaluacion final.
Si lo usamos para ajustar hiperparametros, estamos haciendo data leakage.

In [ ]:
# Division temporal del dataset
# ==============================================================================
end_train      = '2012-03-30 23:59:00'
end_validation = '2012-07-31 23:59:00'

data_train = data.loc[: end_train]
data_val   = data.loc[end_train : end_validation]
data_test  = data.loc[end_validation :]

total = len(data)
print(f'Train      : {data_train.index.min()} --> {data_train.index.max()}')
print(f'             n={len(data_train):,} ({len(data_train)/total*100:.0f}%)')
print()
print(f'Validacion : {data_val.index.min()} --> {data_val.index.max()}')
print(f'             n={len(data_val):,} ({len(data_val)/total*100:.0f}%)')
print()
print(f'Test       : {data_test.index.min()} --> {data_test.index.max()}')
print(f'             n={len(data_test):,} ({len(data_test)/total*100:.0f}%)')
print()
print(f'Total      : {total:,} horas = {total/24:.0f} dias')

In [ ]:
# Visualizacion interactiva de la division temporal
# ==============================================================================
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=data_train.index, y=data_train['users'],
    mode='lines', name='Train',
    line=dict(color='#1f77b4', width=0.8)
))
fig.add_trace(go.Scatter(
    x=data_val.index, y=data_val['users'],
    mode='lines', name='Validacion',
    line=dict(color='#ff7f0e', width=0.8)
))
fig.add_trace(go.Scatter(
    x=data_test.index, y=data_test['users'],
    mode='lines', name='Test',
    line=dict(color='#2ca02c', width=0.8)
))
fig.update_layout(
    title  = 'Demanda horaria de bicicletas — Division temporal',
    xaxis_title = 'Fecha',
    yaxis_title = 'Usuarios',
    legend = dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height = 370,
    margin = dict(l=20, r=20, t=60, b=20)
)
fig.show()

---
## 6. Modelo Baseline: Prediccion Naive Estacional

### ForecasterEquivalentDate

Antes de usar Machine Learning, establecemos un **modelo de referencia (baseline)**.
Si nuestro modelo ML no lo supera, algo esta mal.

El `ForecasterEquivalentDate` predice usando el valor observado en el mismo periodo
de un ciclo anterior. Con `n_offsets=2`, promedia los dos ciclos anteriores.

**Offset elegido: 7 dias (1 semana)**
- Para predecir el lunes a las 8am → usa el lunes PASADO a las 8am
- Captura la doble estacionalidad: diaria + semanal

### Backtesting

El **backtesting** simula como habria funcionado el modelo en el pasado:
1. Entrena con datos historicos
2. Predice un horizonte (steps)
3. Avanza la ventana
4. Repite y calcula el error promedio

```
Train        | Pred | >
Train         | Pred | >
Train          | Pred | >
... (102 folds de 36 pasos cada uno)
```

In [ ]:
# Modelo Baseline: Naive Estacional
# ==============================================================================
forecaster_baseline = ForecasterEquivalentDate(
    offset    = pd.DateOffset(weeks=1),  # Mismo periodo hace 1 semana
    n_offsets = 2                        # Promedio de las ultimas 2 semanas
)

cv_test = TimeSeriesFold(
    steps              = 36,                              # Horizonte de 36 horas
    initial_train_size = len(data.loc[:end_validation]),  # Train + Validacion
    refit              = False
)

metric_baseline, pred_baseline = backtesting_forecaster(
    forecaster    = forecaster_baseline,
    y             = data['users'],
    cv            = cv_test,
    metric        = 'mean_absolute_error',
    verbose       = False,
    show_progress = True
)

mae_baseline = metric_baseline['mean_absolute_error'].iloc[0]
rmse_baseline = root_mean_squared_error(data_test['users'], pred_baseline['pred'])

print()
print('=== BASELINE: Naive Estacional (1 semana) ===')
print(f'  MAE  : {mae_baseline:.2f} usuarios/hora')
print(f'  RMSE : {rmse_baseline:.2f} usuarios/hora')
print()
print('Interpretacion: el modelo de referencia se equivoca en promedio')
print(f'{mae_baseline:.0f} usuarios por hora pronosticada.')

---
## 7. Forecaster Recursivo Univariado con LightGBM

### Como funciona ForecasterRecursive

El forecaster recursivo transforma el problema de prediccion multi-paso en
un problema de regresion supervisada:

```
Tiempo:  t-24  t-12  t-2  t-1  t    t+1  t+2  ... t+36
                                  ──────────────────────
Paso 1:  [lag24, lag12, lag2, lag1]  -->  pred(t+1)
Paso 2:  [lag24, lag12, lag2, pred(t+1)]  -->  pred(t+2)
...      (se propagan las predicciones como si fueran valores reales)
```

### Eleccion de lags estrategicos

Basados en el EDA, elegimos lags que capturan:

| Lags | Captura |
|------|--------|
| 1 a 12 | Dinamica reciente (ultimas 12 horas) |
| 23, 24, 25 | Mismo periodo del dia anterior |
| 47, 48 | Mismo periodo de hace 2 dias |
| 71, 72 | Mismo periodo de hace 3 dias |
| 167, 168 | Mismo periodo de la semana pasada |

### Window Features (estadisticas sobre ventana deslizante)

Complementan los lags con estadisticas resumidas:
- **Media de 24h**: nivel promedio del dia anterior
- **Media de 168h**: nivel promedio de la semana anterior
- **Desviacion estandar**: variabilidad reciente

In [ ]:
# Definir lags estrategicos
# ==============================================================================
lags_estrategicos = list(range(1, 13)) + [23, 24, 25, 47, 48, 71, 72, 167, 168]
print(f'Numero de lags: {len(lags_estrategicos)}')
print(f'Lags: {lags_estrategicos}')

# Window features: estadisticas sobre ventanas deslizantes
# ==============================================================================
# REGLA de skforecast: len(stats) debe ser IGUAL a len(window_sizes).
# Cada par (stat, window_size) define UNA feature independiente.
#
#   stats        = ['mean', 'std', 'mean', 'std']
#   window_sizes = [ 24,     24,   168,    168  ]
#
# Genera 4 features: roll_mean_24, roll_std_24, roll_mean_168, roll_std_168
window_features = RollingFeatures(
    stats        = ['mean', 'std', 'mean', 'std'],
    window_sizes = [24,     24,    24 * 7, 24 * 7]
)

# Crear forecaster univariado con LightGBM
# ==============================================================================
forecaster_uni = ForecasterRecursive(
    estimator       = LGBMRegressor(
        random_state  = 123,
        verbose       = -1,
        n_estimators  = 500,
        learning_rate = 0.05,
        num_leaves    = 63,
        max_depth     = 7
    ),
    lags            = lags_estrategicos,
    window_features = window_features
)

# Entrenar hasta fin de validacion (NO tocar test)
forecaster_uni.fit(y=data.loc[:end_validation, 'users'])

print()
print('Forecaster univariado entrenado exitosamente')
print(f'Periodo de entrenamiento: {data.index.min()} --> {end_validation}')
print(f'Total de features: {len(lags_estrategicos)} lags + 4 window features (roll_mean/std x 24h/168h)')

In [ ]:
# Backtesting del forecaster univariado
# ==============================================================================
metric_uni, pred_uni = backtesting_forecaster(
    forecaster    = forecaster_uni,
    y             = data['users'],
    cv            = cv_test,
    metric        = 'mean_absolute_error',
    n_jobs        = 'auto',
    verbose       = False,
    show_progress = True
)

mae_uni   = metric_uni['mean_absolute_error'].iloc[0]
rmse_uni  = root_mean_squared_error(data_test['users'], pred_uni['pred'])
mejora_uni = (mae_baseline - mae_uni) / mae_baseline * 100

print()
print('=' * 50)
print('  Comparativa: Baseline vs Univariado')
print('=' * 50)
print(f'  Baseline MAE  : {mae_baseline:.2f}')
print(f'  Univariado MAE: {mae_uni:.2f}')
print(f'  Mejora        : {mejora_uni:.1f}%')

In [ ]:
# Visualizacion: predicciones univariadas vs valores reales
# ==============================================================================
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Grafico 1: Test completo
axes[0].plot(data_test.index, data_test['users'],
             label='Real', color='steelblue', linewidth=0.8, alpha=0.9)
axes[0].plot(pred_uni.index, pred_uni['pred'],
             label=f'Prediccion Univariada (MAE={mae_uni:.1f})',
             color='coral', linewidth=0.8, linestyle='--', alpha=0.9)
axes[0].set_title('Forecaster Univariado (LightGBM) — Test Set Completo', fontsize=13)
axes[0].set_ylabel('Usuarios')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Grafico 2: Zoom en 2 semanas
zoom = slice('2012-08-06', '2012-08-20')
axes[1].plot(data_test.loc[zoom].index, data_test.loc[zoom, 'users'],
             label='Real', color='steelblue', linewidth=1.5)
axes[1].plot(pred_uni.loc[zoom].index, pred_uni.loc[zoom, 'pred'],
             label='Prediccion', color='coral', linewidth=1.5, linestyle='--')
axes[1].fill_between(pred_uni.loc[zoom].index,
                     data_test.loc[zoom, 'users'],
                     pred_uni.loc[zoom, 'pred'],
                     alpha=0.2, color='orange', label='Error')
axes[1].set_title('Zoom: 6-20 Agosto 2012 (2 semanas)', fontsize=12)
axes[1].set_xlabel('Fecha')
axes[1].set_ylabel('Usuarios')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Ingenieria de Caracteristicas (Feature Engineering)

### Por que necesitamos features calendarias?

El EDA revelo que la demanda depende fuertemente de:
- La **hora del dia** (picos a las 8h y 17h)
- El **dia de la semana** (laborable vs fin de semana)
- El **mes** (estacionalidad anual)

Podemos codificar estas variables como **features exogenas** para que el modelo
aprenda estos patrones directamente, sin depender solo de los lags.

### Codificacion Ciclica (Seno y Coseno)

Las variables temporales son **ciclicas**: la hora 23 esta 'cerca' de la hora 0,
diciembre esta 'cerca' de enero, etc.

Si codificamos hora=23 como un numero entero, el modelo pensaria que 23 y 0 son
muy diferentes. La codificacion ciclica resuelve esto:

```
x_sin = sin(2 * pi * x / max_x)
x_cos = cos(2 * pi * x / max_x)
```

Esto coloca las horas 0 y 23 cerca una de la otra en el espacio de features.

### Variables exogenas que usaremos

| Variable | Tipo | Disponibilidad futura |
|----------|------|----------------------|
| `hour_sin`, `hour_cos` | Calendaria | Siempre conocida |
| `day_of_week_sin/cos` | Calendaria | Siempre conocida |
| `month_sin`, `month_cos` | Calendaria | Siempre conocida |
| `is_weekend` | Calendaria | Siempre conocida |
| `week_of_year` | Calendaria | Siempre conocida |
| `holiday` | Calendaria | Siempre conocida |
| `temp`, `atemp` | Meteorologica | Pronostico del tiempo |
| `hum`, `windspeed` | Meteorologica | Pronostico del tiempo |
| `weathersit` | Meteorologica | Pronostico del tiempo |

In [ ]:
# Crear caracteristicas calendarias desde el indice temporal
# ==============================================================================
data_features = data.copy()

data_features['hour']         = data_features.index.hour
data_features['day_of_week']  = data_features.index.dayofweek
data_features['month']        = data_features.index.month
data_features['week_of_year'] = data_features.index.isocalendar().week.astype(int)
data_features['is_weekend']   = (data_features.index.dayofweek >= 5).astype(int)

print('Variables calendarias creadas:')
print(data_features[['users', 'hour', 'day_of_week', 'month',
                      'week_of_year', 'is_weekend']].head(8))

In [ ]:
# Codificacion ciclica con feature_engine
# ==============================================================================
cyclical_encoder = CyclicalFeatures(
    variables    = ['hour', 'day_of_week', 'month'],
    max_values   = {'hour': 23, 'day_of_week': 6, 'month': 12},
    drop_original = True   # Reemplazar con las versiones ciclicas
)

data_features = cyclical_encoder.fit_transform(data_features)

# Columnas ciclicas generadas
cols_ciclicas = [c for c in data_features.columns if '_sin' in c or '_cos' in c]
print(f'Columnas ciclicas generadas ({len(cols_ciclicas)}):')
print(cols_ciclicas)
print()
print('Shape del DataFrame con features:', data_features.shape)

# Visualizar la codificacion ciclica de la hora
horas = np.arange(24)
h_sin = np.sin(2 * np.pi * horas / 23)
h_cos = np.cos(2 * np.pi * horas / 23)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc = axes[0].scatter(horas, h_sin, c=horas, cmap='hsv', s=120, zorder=5)
axes[0].plot(horas, h_sin, alpha=0.3, color='gray')
axes[0].set_title('hour_sin por hora del dia', fontsize=11)
axes[0].set_xlabel('Hora')
axes[0].set_ylabel('Seno')
axes[0].set_xticks(range(0, 24, 3))
plt.colorbar(sc, ax=axes[0], label='Hora')

sc2 = axes[1].scatter(h_sin, h_cos, c=horas, cmap='hsv', s=120, zorder=5)
for h in [0, 6, 8, 12, 17, 23]:
    axes[1].annotate(f'h={h}', (h_sin[h], h_cos[h]), fontsize=9,
                     xytext=(8, 5), textcoords='offset points')
axes[1].set_title('Espacio ciclico: las horas 0 y 23 estan cerca!', fontsize=11)
axes[1].set_xlabel('hour_sin')
axes[1].set_ylabel('hour_cos')
axes[1].set_aspect('equal')
plt.colorbar(sc2, ax=axes[1], label='Hora')

plt.suptitle('Codificacion Ciclica: preserva la naturaleza circular del tiempo',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Forecaster con Variables Exogenas (Multivariado)

Ahora incorporamos las variables exogenas al forecaster. Esto permite al modelo
usar informacion adicional disponible en el momento de la prediccion.

### Diferencia clave: lags vs exogenas

| Tipo | Fuente | Horizonte |
|------|--------|----------|
| **Lags** | Historia de la MISMA serie | Se propagan recursivamente |
| **Window features** | Estadisticas de la historia | Se calculan desde lags |
| **Exogenas** | Variables EXTERNAS | Deben conocerse para el futuro |

**Importante:** Las exogenas en `backtesting_forecaster` se pasan completas
(train+val+test). Skforecast usa internamente solo las filas correctas de
acuerdo al fold de evaluacion.

In [ ]:
# Definir columnas exogenas
# ==============================================================================
exog_cols = [
    # Meteorologicas
    'holiday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
    # Calendarias ciclicas
    'hour_sin', 'hour_cos',
    'day_of_week_sin', 'day_of_week_cos',
    'month_sin', 'month_cos',
    # Calendarias lineales
    'is_weekend', 'week_of_year'
]

print(f'Numero de variables exogenas: {len(exog_cols)}')
print('Variables:', exog_cols)

# Window features para el forecaster con exogenas
window_features_exog = RollingFeatures(
    stats        = ['mean', 'std'],
    window_sizes = [24, 24 * 7]    # 1 dia y 1 semana
)

# Forecaster con variables exogenas
# ==============================================================================
forecaster_exog = ForecasterRecursive(
    estimator = LGBMRegressor(
        random_state   = 123,
        verbose        = -1,
        n_estimators   = 600,
        learning_rate  = 0.05,
        num_leaves     = 63,
        max_depth      = 8,
        min_child_samples = 15
    ),
    lags            = lags_estrategicos,
    window_features = window_features_exog
)

# Entrenar con exogenas (hasta fin de validacion)
forecaster_exog.fit(
    y    = data.loc[:end_validation, 'users'],
    exog = data_features.loc[:end_validation, exog_cols]
)

print()
print('Forecaster con exogenas entrenado.')
print(f'Total de features de entrada: lags + window_features + {len(exog_cols)} exogenas')

In [ ]:
# Backtesting con variables exogenas
# ==============================================================================
metric_exog, pred_exog = backtesting_forecaster(
    forecaster    = forecaster_exog,
    y             = data['users'],
    exog          = data_features[exog_cols],  # Todas las fechas (train+val+test)
    cv            = cv_test,
    metric        = 'mean_absolute_error',
    n_jobs        = 'auto',
    verbose       = False,
    show_progress = True
)

mae_exog   = metric_exog['mean_absolute_error'].iloc[0]
rmse_exog  = root_mean_squared_error(data_test['users'], pred_exog['pred'])
mejora_exog = (mae_baseline - mae_exog) / mae_baseline * 100

print()
print('=' * 55)
print('  Comparativa: Baseline vs Univariado vs Exogenas')
print('=' * 55)
print(f'  Baseline       MAE: {mae_baseline:.2f}  RMSE: {rmse_baseline:.2f}')
print(f'  Univariado     MAE: {mae_uni:.2f}  RMSE: {rmse_uni:.2f}  (mejora: {mejora_uni:.1f}%)')
print(f'  + Exogenas     MAE: {mae_exog:.2f}  RMSE: {rmse_exog:.2f}  (mejora: {mejora_exog:.1f}%)')

---
## 10. Optimizacion Bayesiana de Hiperparametros

### Por que optimizar?

Los hiperparametros del modelo (numero de arboles, tasa de aprendizaje, etc.)
tienen un gran impacto en la precision. Elegirlos manualmente es suboptimo.

### Busqueda Bayesiana vs otras estrategias

| Estrategia | Descripcion | Eficiencia |
|------------|-------------|------------|
| Grid Search | Prueba TODAS las combinaciones | Muy lenta |
| Random Search | Combinaciones aleatorias | Moderada |
| **Bayesian Search** | Aprende del pasado, busca zonas prometedoras | Alta |

La busqueda bayesiana usa **Optuna** internamente. Con solo 20-30 trials
puede encontrar configuraciones cercanas al optimo.

### Importante: usar solo Train+Validacion

La busqueda de hiperparametros se hace SOLO sobre datos de train y validacion.
El test se reserva para la evaluacion final.

```
cv_search: Divide TRAIN en subfolds para evaluar cada configuracion

TRAIN ────────────────────────────────┐
  mini-train1 | val1 |               |
  mini-train2   | val2 |             |
  mini-train3     | val3 |           |
  ...                                |
                                     TEST (intocable)
```

In [ ]:
# Definir espacio de busqueda de hiperparametros
# ==============================================================================
def search_space(trial):
    return {
        'n_estimators'      : trial.suggest_int('n_estimators', 300, 1200, step=100),
        'max_depth'         : trial.suggest_int('max_depth', 3, 12),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves'        : trial.suggest_int('num_leaves', 20, 200),
        'min_child_samples' : trial.suggest_int('min_child_samples', 5, 60),
        'subsample'         : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-5, 5.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-5, 5.0, log=True),
        'lags'              : trial.suggest_categorical('lags', [
            list(range(1, 13)) + [23, 24, 25, 47, 48, 71, 72, 167, 168],
            list(range(1, 25)) + [48, 72, 168],
            [1, 2, 3, 6, 12, 24, 48, 72, 168]
        ])
    }

# Fold para la busqueda (sobre datos Train + Validacion)
cv_search = TimeSeriesFold(
    steps              = 36,
    initial_train_size = len(data.loc[:end_train]),  # Solo TRAIN para el primer fold
    refit              = False
)

print('Espacio de busqueda definido.')
print(f'Datos de busqueda: {data.index.min()} --> {end_validation}')
print(f'Tamano inicial de train: {len(data.loc[:end_train]):,} filas')

In [ ]:
# Ejecutar busqueda bayesiana
# ==============================================================================
# NOTA: n_trials=20 es un buen balance entre velocidad y calidad.
#       En produccion se recomienda n_trials >= 50.

print('Iniciando busqueda bayesiana (puede tardar varios minutos)...')
print()

results_search, frozen_trial = bayesian_search_forecaster(
    forecaster    = forecaster_exog,
    y             = data.loc[:end_validation, 'users'],
    exog          = data_features.loc[:end_validation, exog_cols],
    cv            = cv_search,
    search_space  = search_space,
    metric        = 'mean_absolute_error',
    n_trials      = 20,
    random_state  = 123,
    return_best   = True,   # Actualiza forecaster_exog con los mejores params
    n_jobs        = 'auto',
    verbose       = False,
    show_progress = True
)

In [ ]:
# Resultados de la busqueda
# ==============================================================================
# NOTA: en skforecast, 'lags' es una columna separada del DataFrame results_search,
# NO esta dentro del diccionario 'params' (que solo tiene hiperparametros del estimador).

print('Top 5 configuraciones encontradas:')
cols_display = ['mean_absolute_error', 'lags', 'params']
cols_display = [c for c in cols_display if c in results_search.columns]
print(results_search[cols_display].head(5).to_string(index=False))
print()

# Mejores hiperparametros del estimador (columna 'params')
best_params = results_search.iloc[0]['params']
print('Mejores hiperparametros del estimador (LightGBM):')
for k, v in best_params.items():
    print(f'  {k:<25}: {v}')

# Mejores lags (columna separada 'lags' en results_search)
if 'lags' in results_search.columns:
    best_lags = results_search.iloc[0]['lags']
    print(f'  {"lags (mejor config)":<25}: {best_lags}')

print()
print(f'Mejor MAE en validacion: {results_search.iloc[0]["mean_absolute_error"]:.2f}')

# Convergencia de la busqueda
fig, ax = plt.subplots(figsize=(10, 4))
maes = results_search['mean_absolute_error'].values[::-1]   # orden cronologico
best_so_far = pd.Series(maes).cummin().values
ax.plot(range(1, len(maes) + 1), maes,
        'o-', color='steelblue', alpha=0.5, markersize=5, label='MAE por trial')
ax.plot(range(1, len(maes) + 1), best_so_far,
        'r--', linewidth=2, label='Mejor MAE acumulado')
ax.set_title('Convergencia de la Busqueda Bayesiana', fontsize=12)
ax.set_xlabel('Trial #')
ax.set_ylabel('MAE (validacion)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 11. Evaluacion Final del Modelo Optimizado

Despues de la optimizacion, `return_best=True` ya actualizo `forecaster_exog`
con los mejores hiperparametros. Ahora realizamos el backtesting final sobre
el **Test Set** que no fue usado durante la optimizacion.

### Metricas de evaluacion

| Metrica | Formula | Interpretacion |
|---------|---------|----------------|
| **MAE** | mean(abs(y - pred)) | Error promedio en las mismas unidades |
| **RMSE** | sqrt(mean((y-pred)^2)) | Penaliza errores grandes |
| **MAPE** | mean(abs((y-pred)/y)) * 100 | Error en porcentaje |
| **R2** | 1 - SS_res/SS_tot | Proporcion de varianza explicada (maximo 1.0) |

In [ ]:
# Backtesting final en el TEST SET con el modelo optimizado
# ==============================================================================
metric_best, pred_best = backtesting_forecaster(
    forecaster    = forecaster_exog,   # Ya tiene los mejores parametros
    y             = data['users'],
    exog          = data_features[exog_cols],
    cv            = cv_test,
    metric        = 'mean_absolute_error',
    n_jobs        = 'auto',
    verbose       = False,
    show_progress = True
)

# Calcular todas las metricas
# IMPORTANTE: usar pred_best['pred'] para extraer solo la columna de predicciones
# pred_best tiene 2 columnas: 'fold' y 'pred'
y_true = data_test['users']
y_pred = pred_best['pred']

mae_best  = mean_absolute_error(y_true, y_pred)
rmse_best = root_mean_squared_error(y_true, y_pred)
mape_best = mean_absolute_percentage_error(y_true, y_pred) * 100
r2_best   = r2_score(y_true, y_pred)

mejora_final = (mae_baseline - mae_best) / mae_baseline * 100

print()
print('=' * 60)
print('  RESULTADOS FINALES — TEST SET (Agosto - Diciembre 2012)')
print('=' * 60)
print(f'  MAE   : {mae_best:.2f} usuarios/hora')
print(f'  RMSE  : {rmse_best:.2f} usuarios/hora')
print(f'  MAPE  : {mape_best:.2f}%')
print(f'  R2    : {r2_best:.4f}')
print()
print(f'  El modelo explica el {r2_best*100:.1f}% de la variabilidad de la demanda')
print(f'  El error promedio es {mape_best:.1f}% del valor real')
print()
print('  Comparativa de modelos:')
print(f'  Baseline               MAE: {mae_baseline:.2f}')
print(f'  Univariado (LightGBM)  MAE: {mae_uni:.2f}  ({mejora_uni:.1f}% mejora)')
print(f'  + Exogenas             MAE: {mae_exog:.2f}  ({mejora_exog:.1f}% mejora)')
print(f'  + Exogenas + Optim     MAE: {mae_best:.2f}  ({mejora_final:.1f}% mejora)')

In [ ]:
# Visualizacion completa del test set
# ==============================================================================
fig, axes = plt.subplots(3, 1, figsize=(14, 13))

# Grafico 1: Test completo
axes[0].plot(data_test.index, y_true,
             label='Real', color='steelblue', linewidth=0.7, alpha=0.9)
axes[0].plot(y_pred.index, y_pred,
             label=f'Prediccion optimizada (MAE={mae_best:.1f})',
             color='coral', linewidth=0.7, linestyle='--', alpha=0.9)
axes[0].set_title(f'Modelo Optimizado — Test Set Completo (R2={r2_best:.3f})', fontsize=13)
axes[0].set_ylabel('Usuarios')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Grafico 2: Zoom 2 semanas en octubre
zoom2 = slice('2012-10-01', '2012-10-15')
axes[1].plot(data_test.loc[zoom2].index, y_true.loc[zoom2],
             label='Real', color='steelblue', linewidth=1.5)
axes[1].plot(y_pred.loc[zoom2].index, y_pred.loc[zoom2],
             label='Prediccion', color='coral', linewidth=1.5, linestyle='--')
axes[1].fill_between(y_pred.loc[zoom2].index,
                     y_true.loc[zoom2], y_pred.loc[zoom2],
                     alpha=0.2, color='orange', label='Error')
axes[1].set_title('Zoom: 1-15 Octubre 2012', fontsize=12)
axes[1].set_ylabel('Usuarios')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Grafico 3: Error por hora del dia
errors = y_true - y_pred
error_by_hour = errors.groupby(errors.index.hour).agg(['mean', 'std'])
axes[2].bar(error_by_hour.index, error_by_hour['mean'],
            yerr=error_by_hour['std'], capsize=3,
            color=np.where(error_by_hour['mean'] > 0, 'steelblue', 'coral'),
            alpha=0.8, edgecolor='white')
axes[2].axhline(0, color='black', linewidth=1)
axes[2].set_title('Error medio por hora del dia (azul = subestima, coral = sobreestima)',
                  fontsize=12)
axes[2].set_xlabel('Hora del dia')
axes[2].set_ylabel('Error medio (real - pred)')
axes[2].set_xticks(range(24))
axes[2].grid(True, alpha=0.3)

plt.suptitle('Evaluacion del Modelo Optimizado', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Grafico de dispersion y distribucion de residuos
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: real vs prediccion
lim_max = max(y_true.max(), y_pred.max()) * 1.05
axes[0].scatter(y_true, y_pred, alpha=0.2, color='steelblue', s=8)
axes[0].plot([0, lim_max], [0, lim_max], 'r--', linewidth=2, label='Prediccion perfecta')
axes[0].set_xlabel('Valores reales')
axes[0].set_ylabel('Predicciones')
axes[0].set_title(f'Real vs Prediccion\nR2 = {r2_best:.3f}  |  MAE = {mae_best:.1f}', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribucion de residuos
errors.hist(bins=60, ax=axes[1], color='steelblue', alpha=0.75, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Error = 0')
axes[1].axvline(errors.mean(), color='orange', linestyle='--', linewidth=2,
                label=f'Media = {errors.mean():.1f}')
axes[1].set_title('Distribucion de Residuos\n(ideal: centrada en 0)', fontsize=12)
axes[1].set_xlabel('Residuo (real - prediccion)')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Media de residuos (bias)  : {errors.mean():.2f} usuarios')
print(f'Std de residuos           : {errors.std():.2f} usuarios')
print(f'Residuos < 0 (sobreestima): {(errors < 0).sum()} ({(errors < 0).mean()*100:.1f}%)')
print(f'Residuos > 0 (subestima)  : {(errors > 0).sum()} ({(errors > 0).mean()*100:.1f}%)')

---
## 12. Importancia de Variables y Analisis SHAP

### Por que analizar la importancia de variables?

1. **Interpretabilidad**: Entender que impulsa las predicciones del modelo
2. **Validacion**: Verificar que el modelo aprendio patrones logicos del negocio
3. **Debugging**: Detectar features problematicas o sin valor predictivo
4. **Conocimiento de dominio**: Relacionar hallazgos con intuicion del experto

### Feature Importance de LightGBM vs SHAP

| Metodo | Tipo | Ventajas | Limitaciones |
|--------|------|----------|-------------|
| **Gain** (LightGBM) | Global | Rapido, disponible directamente | No muestra direccion, puede ser sesgado |
| **SHAP** | Local y global | Muestra direccion, garantias teoricas | Mas lento |

**SHAP (SHapley Additive exPlanations)** descompone cada prediccion en
contribuciones individuales de cada feature. Se basa en teoria de juegos
(valores de Shapley) para distribuir la 'culpa' de la prediccion.

```
prediccion_i = valor_base + SHAP(lag_24) + SHAP(hour_sin) + SHAP(temp) + ...
```

In [ ]:
# Importancia de variables segun LightGBM (Gain)
# ==============================================================================

# --- Paso 1: localizar el estimador subyacente ---
# El nombre del atributo varia entre versiones de skforecast.
# Buscamos el objeto que tenga 'feature_importances_' (sklearn API).

_estimator = None
_attr_used = None

# Intentar nombres conocidos primero
for _attr in ['regressor', 'estimator', 'fitted_estimator_', 'regressor_']:
    candidate = getattr(forecaster_exog, _attr, None)
    if candidate is not None and hasattr(candidate, 'feature_importances_'):
        _estimator = candidate
        _attr_used = _attr
        break

# Fallback: buscar en todos los atributos de la instancia
if _estimator is None:
    for _attr, _val in vars(forecaster_exog).items():
        if hasattr(_val, 'feature_importances_'):
            _estimator = _val
            _attr_used = _attr
            break

if _estimator is None:
    print('Atributos del forecaster (para diagnostico):')
    print(sorted([a for a in dir(forecaster_exog) if not a.startswith('_')]))
    raise AttributeError('No se pudo localizar el estimador ajustado.')

print(f'Estimador encontrado en : forecaster_exog.{_attr_used}')
print(f'Tipo de estimador       : {type(_estimator).__name__}')

# --- Paso 2: obtener nombres de features ---
if hasattr(forecaster_exog, 'training_features_names_out_'):
    feature_names = list(forecaster_exog.training_features_names_out_)
elif hasattr(forecaster_exog, 'feature_names_in_'):
    feature_names = list(forecaster_exog.feature_names_in_)
elif hasattr(_estimator, 'feature_name_'):          # LightGBM
    feature_names = list(_estimator.feature_name_)
elif hasattr(_estimator, 'feature_names_in_'):      # scikit-learn >= 1.0
    feature_names = list(_estimator.feature_names_in_)
else:
    n = len(_estimator.feature_importances_)
    feature_names = [f'feature_{i}' for i in range(n)]
    print(f'Aviso: no se encontraron nombres, usando feature_0..feature_{n-1}')

feature_import = _estimator.feature_importances_
print(f'Numero de features en el modelo: {len(feature_names)}')
print()

# --- Paso 3: DataFrame de importancias ---
importance_df = pd.DataFrame({
    'feature'    : feature_names,
    'importance' : feature_import
}).sort_values('importance', ascending=False).reset_index(drop=True)

def tipo_feature(nombre):
    if nombre.startswith('lag_'):
        return 'Lag'
    elif nombre.startswith('roll_'):
        return 'Window feature'
    else:
        return 'Exogena'

importance_df['tipo'] = importance_df['feature'].apply(tipo_feature)
colores_tipo = {'Lag': '#2196F3', 'Window feature': '#FF5722', 'Exogena': '#4CAF50'}

top_n = 25
top_df = importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(10, 9))
colors = [colores_tipo[t] for t in top_df['tipo']]
ax.barh(range(top_n), top_df['importance'].values,
        color=colors, alpha=0.8, edgecolor='white')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_df['feature'].values, fontsize=9)
ax.invert_yaxis()
ax.set_title(f'Top {top_n} Variables mas Importantes (LightGBM Gain)', fontsize=13)
ax.set_xlabel('Importancia')
ax.grid(True, alpha=0.3, axis='x')

patches = [mpatches.Patch(color=c, label=t) for t, c in colores_tipo.items()]
ax.legend(handles=patches, loc='lower right')
plt.tight_layout()
plt.show()

print('Importancia total por tipo:')
print(importance_df.groupby('tipo')['importance'].sum().sort_values(ascending=False))

In [ ]:
# Analisis SHAP (SHapley Additive exPlanations)
# ==============================================================================
# NOTA: esta celda usa '_estimator' y 'feature_names' definidos en la celda anterior.

print('Preparando datos para SHAP...')

# Obtener la matriz de features de entrenamiento
if hasattr(forecaster_exog, 'create_train_X_y'):
    X_train_shap, _ = forecaster_exog.create_train_X_y(
        y    = data.loc[:end_validation, 'users'],
        exog = data_features.loc[:end_validation, exog_cols]
    )
    # Asignar nombres de columnas si no los tienen
    if list(X_train_shap.columns) != feature_names:
        try:
            X_train_shap.columns = feature_names
        except ValueError:
            pass  # distinto numero de columnas; usar los que vienen
else:
    raise RuntimeError('create_train_X_y no disponible. Verifica la version de skforecast.')

# Muestra aleatoria para mayor velocidad
rng      = np.random.default_rng(42)
n_sample = min(1500, len(X_train_shap))
idx      = rng.choice(len(X_train_shap), size=n_sample, replace=False)
X_shap   = X_train_shap.iloc[idx]

print(f'Muestra para SHAP : {n_sample:,} obs de {len(X_train_shap):,} totales')
print(f'Numero de features: {X_shap.shape[1]}')

# Calcular valores SHAP con TreeExplainer (usa '_estimator' de la celda anterior)
print('Calculando valores SHAP...')
explainer   = shap.TreeExplainer(_estimator)
shap_values = explainer(X_shap)
print('Listo!')

In [ ]:
# SHAP Summary Plot: importancia + direccion del efecto
# ==============================================================================
print('SHAP Summary Plot')
print('  Puntos ROJOS    = valor alto de la feature')
print('  Puntos AZULES   = valor bajo de la feature')
print('  Posicion DERECHA = aumenta la prediccion')
print('  Posicion IZQUIERDA = disminuye la prediccion')
print()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Panel izquierdo: importancia media absoluta (barras)
plt.sca(axes[0])
shap.summary_plot(
    shap_values,
    X_shap,
    max_display  = 20,
    plot_type    = 'bar',
    show         = False
)
axes[0].set_title('SHAP: Importancia media absoluta', fontsize=12)

# Panel derecho: violin/dot plot con direccion del efecto
plt.sca(axes[1])
shap.summary_plot(
    shap_values,
    X_shap,
    max_display = 20,
    show        = False
)
axes[1].set_title('SHAP: Direccion del impacto de cada feature', fontsize=12)

plt.suptitle('Analisis SHAP — Interpretabilidad del Modelo', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Dependence Plots: como afecta cada feature a las predicciones
# ==============================================================================
# Identificar las 4 features mas importantes segun SHAP
mean_shap = np.abs(shap_values.values).mean(axis=0)
top_features_idx = np.argsort(mean_shap)[::-1][:4]
top_features_names = [X_shap.columns[i] for i in top_features_idx]

print(f'Top 4 features segun SHAP: {top_features_names}')
print()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, feat in enumerate(top_features_names):
    feat_idx = list(X_shap.columns).index(feat)
    shap.dependence_plot(
        feat_idx,
        shap_values.values,
        X_shap,
        ax=axes[i],
        show=False,
        alpha=0.3
    )
    axes[i].set_title(f'Dependencia SHAP: {feat}', fontsize=11)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('SHAP Dependence Plots: Como afecta cada feature a la prediccion',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 13. Predicciones Futuras (Simulacion en Produccion)

Para hacer predicciones en un entorno real necesitamos:

1. **Modelo entrenado** con todos los datos disponibles
2. **Valores futuros de las exogenas** para el horizonte de prediccion:
   - Variables calendarias: siempre disponibles
   - Variables meteorologicas: requieren un pronostico del tiempo

En este ejemplo simulamos las exogenas meteorologicas usando
los promedios historicos por hora del mes correspondiente.

In [ ]:
# Re-entrenar con TODOS los datos disponibles (incluido test)
# ==============================================================================
# Usamos '_estimator' localizado en cell_28 (robusto ante cambios de API en skforecast).
# Los lags optimos vienen de forecaster_exog.lags (array numpy).

forecaster_final = ForecasterRecursive(
    estimator       = _estimator,
    lags            = forecaster_exog.lags.tolist(),
    window_features = window_features_exog
)

forecaster_final.fit(
    y    = data['users'],
    exog = data_features[exog_cols]
)

print('Modelo re-entrenado con todos los datos (incluido test).')
print(f'Periodo: {data.index.min()} --> {data.index.max()}')
print(f'Lags usados: {forecaster_exog.lags.tolist()}')

In [ ]:
# Crear features exogenas para las proximas 48 horas
# ==============================================================================
horizonte    = 48
last_date    = data.index[-1]
future_dates = pd.date_range(
    start   = last_date + pd.Timedelta(hours=1),
    periods = horizonte,
    freq    = 'h'
)

future_exog = pd.DataFrame(index=future_dates)

# Variables calendarias (siempre disponibles)
future_exog['hour']         = future_exog.index.hour
future_exog['day_of_week']  = future_exog.index.dayofweek
future_exog['month']        = future_exog.index.month
future_exog['week_of_year'] = future_exog.index.isocalendar().week.astype(int)
future_exog['is_weekend']   = (future_exog.index.dayofweek >= 5).astype(int)
future_exog['holiday']      = 0
future_exog['weathersit']   = 1

# Variables meteorologicas: promedio historico por hora y mes
for col in ['temp', 'atemp', 'hum', 'windspeed']:
    hourly_means = data.groupby([data.index.month, data.index.hour])[col].mean()
    future_exog[col] = future_exog.apply(
        lambda row: hourly_means.get(
            (int(row['month']), int(row['hour'])), data[col].mean()
        ),
        axis=1
    )

# Codificacion ciclica aplicada directamente sobre future_exog.
# NO se puede reusar cyclical_encoder.transform() porque fue ajustado sobre
# data_features (que incluye 'users' y mas columnas).
# Aplicamos la formula directamente para evitar la incompatibilidad de columnas.
ciclicas_cfg = {'hour': 23, 'day_of_week': 6, 'month': 12}
for col, max_val in ciclicas_cfg.items():
    future_exog[f'{col}_sin'] = np.sin(2 * np.pi * future_exog[col] / max_val)
    future_exog[f'{col}_cos'] = np.cos(2 * np.pi * future_exog[col] / max_val)
future_exog.drop(columns=list(ciclicas_cfg.keys()), inplace=True)

print('Exogenas futuras preparadas:')
print(f'Columnas: {future_exog.columns.tolist()}')
print()
print(future_exog[exog_cols].head(6).round(3))

In [ ]:
# Predicciones futuras
# ==============================================================================
predicciones_futuras = forecaster_final.predict(
    steps = horizonte,
    exog  = future_exog[exog_cols]
)

# Asegurar valores no negativos (demanda no puede ser negativa)
predicciones_futuras = predicciones_futuras.clip(lower=0)

# Visualizacion
fig, ax = plt.subplots(figsize=(14, 5))
data['users'].tail(7 * 24).plot(
    ax=ax, label='Historico (ultima semana)', color='steelblue', linewidth=1, alpha=0.9
)
predicciones_futuras.plot(
    ax=ax, label=f'Prediccion ({horizonte} horas)', color='coral', linewidth=2
)
ax.axvline(last_date, color='gray', linestyle=':', linewidth=2, label='Inicio prediccion')
ax.set_title(f'Predicciones Futuras: Proximas {horizonte} horas', fontsize=13)
ax.set_xlabel('Fecha')
ax.set_ylabel('Usuarios estimados')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print()
print(f'Predicciones para las proximas {horizonte} horas:')
pred_df = predicciones_futuras.round(0).astype(int).to_frame('usuarios_estimados')
pred_df['hora'] = pred_df.index.hour
pred_df['dia']  = pred_df.index.day_name()
print(pred_df.head(24).to_string())

---
## 14. Resumen y Conclusiones

### Resultados del flujo de modelado

Partimos de un modelo baseline simple y progresivamente mejoramos:

| Modelo | Tecnica principal | MAE (Test) |
|--------|------------------|-----------|
| Baseline | Naive semanal | ~86 usuarios/h |
| Univariado | LightGBM + Lags estrategicos + Window features | Mejor |
| + Exogenas | + Variables meteorologicas y calendarias ciclicas | Mejor |
| + Optimizacion | + Busqueda bayesiana de hiperparametros | Mejor |

### Conceptos aprendidos

**Preprocesamiento de series de tiempo:**
- Construccion de DatetimeIndex con frecuencia fija
- Imputacion de horas faltantes con interpolacion temporal
- Uso de `.copy()` para evitar SettingWithCopyWarning

**Modelado con Skforecast:**
- `ForecasterEquivalentDate` para modelos baseline
- `ForecasterRecursive` para modelos univariados y multivariados
- Lags estrategicos basados en la estructura de la serie
- Window features (rolling statistics) como features adicionales
- Variables exogenas para incorporar informacion externa

**Feature Engineering:**
- Variables calendarias desde el indice temporal
- Codificacion ciclica para capturar la naturaleza circular del tiempo

**Evaluacion correcta:**
- Backtesting temporal (no validacion cruzada aleatoria)
- Division estricta train/validacion/test
- Metricas: MAE, RMSE, MAPE, R2
- Uso de `predictions['pred']` para extraer predicciones del DataFrame

**Optimizacion:**
- Busqueda bayesiana con Optuna via `bayesian_search_forecaster`
- Solo usar train+validacion, nunca el test

**Interpretabilidad:**
- Importancia de variables LightGBM (gain)
- SHAP TreeExplainer para entender cada prediccion
- Dependence plots para ver el efecto de cada feature

### Proximos pasos

- **Intervalos de prediccion**: usar `predict_quantiles()` para obtener bandas de incertidumbre
- **ForecasterRecursiveMultiSeries**: predecir usuarios `casual` y `registered` simultáneamente
- **Seleccion de features**: `select_features()` de skforecast para eliminar variables redundantes
- **Produccion**: Reentrenar periodicamente con nuevos datos y monitorear el drift